# Phase 5 — Evaluation & SHAP Interpretability


In [ ]:
# Phase 5 — Evaluation & SHAP Interpretability

# ============================================================
# Phase 5 — Evaluation and interpretation
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_auc_score, roc_curve)
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

import shap

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✅ Libraries ready")

# ============================================================
# Rebuild, train best model on train, evaluate on test
# ============================================================
DATA_FILE = Path("/kaggle/input/datasets/stephanmatzka/predictive-maintenance-dataset-ai4i-2020/ai4i2020.csv")
df = pd.read_csv(DATA_FILE)

SENSOR_COLUMNS = [
    "Air temperature [K]", "Process temperature [K]",
    "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]",
]
BINARY_TARGET = "Machine failure"
FAILURE_TYPE_COLUMNS = ["TWF", "HDF", "PWF", "OSF", "RNF"]
CATEGORICAL_COL = "Type"

flag_sum = df[FAILURE_TYPE_COLUMNS].sum(axis=1)
df["Failure Type"] = "No Failure"
failed_mask = df[BINARY_TARGET] == 1
df.loc[failed_mask, "Failure Type"] = (
    df.loc[failed_mask, FAILURE_TYPE_COLUMNS]
    .apply(lambda row: "+".join(row.index[row == 1]) or "Unknown", axis=1)
)

df["Power [W]"] = df["Torque [Nm]"] * df["Rotational speed [rpm]"] * (2 * np.pi / 60)
df["Temp Diff [K]"] = df["Process temperature [K]"] - df["Air temperature [K]"]
df["Overstrain [min·Nm]"] = df["Tool wear [min]"] * df["Torque [Nm]"]
ENGINEERED_COLUMNS = ["Power [W]", "Temp Diff [K]", "Overstrain [min·Nm]"]

feature_cols = SENSOR_COLUMNS + ENGINEERED_COLUMNS + [CATEGORICAL_COL]
X = df[feature_cols]
y = df[BINARY_TARGET]

numeric_cols = [c for c in X.columns if X[c].dtype in ["int64", "float64"]]
categorical_cols = [c for c in X.columns if X[c].dtype == "object"]

numeric_transformer = Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ordinal", OrdinalEncoder(categories=[["L", "M", "H"]])),
])
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y,
)

SCALE_POS_WEIGHT = (y_train == 0).sum() / (y_train == 1).sum()

best_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=4,
        scale_pos_weight=SCALE_POS_WEIGHT, random_state=RANDOM_STATE,
        eval_metric="logloss")),
])

best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

print("✅ Training and evaluation done")

# ============================================================
# Confusion matrix
# ============================================================
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                fontsize=20, fontweight="bold",
                color="white" if cm[i, j] > cm.max()/2 else "black")

ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(["Predicted: Healthy", "Predicted: Failed"])
ax.set_yticklabels(["Actual: Healthy", "Actual: Failed"])
ax.set_title("Confusion Matrix", fontsize=13, fontweight="bold")
fig.colorbar(im)
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Positive (TP)  : {tp}")
print(f"False Positive (FP) : {fp}")
print(f"False Negative (FN) ⚠️ : {fn}")
print(f"True Negative (TN)  : {tn}")

# ============================================================
# Classification report + ROC-AUC curve
# ============================================================
print("=" * 55)
print("📋 Classification Report (on test set)")
print("=" * 55)
print(classification_report(y_test, y_pred,
                            target_names=["Healthy (0)", "Failed (1)"],
                            digits=4))

y_proba = best_model.predict_proba(X_test)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color="#4C72B0", lw=2, label=f"AUC = {auc:.4f}")
ax.plot([0, 1], [0, 1], color="gray", linestyle="--", label="Random")
ax.set_xlabel("False Positive Rate (FPR)")
ax.set_ylabel("True Positive Rate (TPR / Recall)")
ax.set_title("ROC Curve", fontsize=13, fontweight="bold")
ax.legend(loc="lower right")
plt.show()

# ============================================================
# Physical interpretation with SHAP
# ============================================================
xgb_clf = best_model.named_steps["classifier"]
preproc = best_model.named_steps["preprocessor"]
X_train_t = preproc.transform(X_train)

clean_names = [
    "Air Temp", "Process Temp", "Rot. Speed", "Torque", "Tool Wear",
    "Power", "Temp Diff", "Overstrain", "Type (Quality)",
]

explainer = shap.TreeExplainer(xgb_clf)
shap_values = explainer.shap_values(X_train_t)

shap.summary_plot(shap_values, X_train_t, feature_names=clean_names)
